# Workshop 02: Time Series Decomposition & Feature Engineering
*Advanced Machine Learning II (ML_II) — Master of Science in Data Science & AI*
*Lazarski University*

---

### Executive Overview & Lab Objectives
In real-world econometric, financial, and industrial applications, raw time series are rarely amenable to direct statistical modeling. They are confounded by calendar anomalies (varying month lengths, leap years), inflationary drift, level-dependent seasonal variance, and non-linear trends.

This workshop provides an end-to-end, mathematically rigorous foundation in **classical and modern time series decomposition**, **variance stabilization**, and **time-series feature engineering**. Over the next 120 minutes, you will build production-grade implementations of:

1. **Calendar & Price Adjustments:** Normalizing monthly variation and deflating price inflation using the FRED Consumer Price Index.
2. **Guerrero Variance Stabilization:** Implementing Guerrero's (1993) algorithm to optimize Box-Cox $\lambda$ and applying second-order Taylor bias corrections.
3. **Classical Centered Moving Averages:** Constructing $2 \times m$-MA filters and understanding the critical structural hazard of **endpoint voids**.
4. **Multiplicative Log-STL Decomposition (Direction 1):** Implementing LOESS-based decomposition on the log-scale to guarantee exact multiplicative reconstruction:
   $$Y_t \equiv T_t \times S_t \times R_t$$
5. **FEASTS Feature Engineering:** Compressing long historical time series into scalar diagnostic feature vectors ($F_T$, $F_S$, peak/trough timing, remainder autocorrelation).
6. **Production STLF Forecaster & Temporal Hygiene Auditing (Direction 4):** Building a seasonal-trend decomposition baseline combining LOESS, drift, and seasonal naïve projection while rigorously auditing data leakage.
7. **Quantitative & Conceptual Interview Questions:** Addressing foundational ML interview questions on Jensen's inequality, boundary extrapolation, and lookahead bias.


In [ ]:
# =============================================================================
# Environment & Library Setup
# =============================================================================
import os
import sys
from pathlib import Path
from typing import Dict, List, Any, Union, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize_scalar
from statsmodels.tsa.seasonal import STL

# Configure visualization styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Helvetica'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path("../data")
RANDOM_SEED = 42
print("Environment configured successfully. Ready for Workshop 02.")


---
## Part 1: Transformations & Adjustments (Calendar & Price Deflation) (20 Minutes)

### Theoretical Foundations
A raw monthly series $Y_t$ (such as retail sales) fluctuates due to two primary non-structural calendar artifacts:
1. **Unequal Month Lengths:** A month can have 28, 29, 30, or 31 days. A 31-day month naturally accumulates $\approx 10.7\%$ more sales volume than February (28 days) purely due to calendar duration, not underlying economic acceleration. Leap years introduce further non-periodic distortions.
2. **Price Inflation:** Nominal dollar figures reflect both physical transaction volume and the declining purchasing power of the currency. To isolate true physical demand, nominal figures must be deflated by a price index:
   $$\text{Real Sales}_t = \frac{\text{Nominal Sales}_t}{\text{CPI}_t} \times \text{CPI}_{\text{base}}$$
   $$\text{Real Sales per Day}_t = \frac{\text{Real Sales}_t}{\text{Days in Month}_t}$$

### Implementation Task 1
Implement `load_and_index_series` and `apply_calendar_and_price_adjustments`.


In [ ]:
def load_and_index_series(
    filepath: str,
    date_col: str,
    val_col: str,
    freq: str = "MS"
) -> pd.Series:
    """
    Ingests a raw time series CSV, enforces an immutable DatetimeIndex and calendar frequency.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


def apply_calendar_and_price_adjustments(
    sales: pd.Series,
    cpi: pd.Series,
    base_cpi: Optional[float] = None
) -> pd.DataFrame:
    """
    Removes calendar artifacts (varying days per month/leap years) and price inflation.

    Parameters
    ----------
    sales : pd.Series
        Nominal sales series with monthly DatetimeIndex.
    cpi : pd.Series
        Consumer Price Index series with monthly DatetimeIndex.
    base_cpi : float, optional
        Base CPI level for deflation. If None, defaults to the last available CPI observation.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
        ['raw_sales', 'cpi', 'real_sales', 'days_in_month', 'real_sales_per_day']
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 1: Calendar and Price Normalization
raw_sales = load_and_index_series(str(DATA_DIR / "RSAFS.csv"), "observation_date", "RSAFSNA", "MS")
cpi_series = load_and_index_series(str(DATA_DIR / "CPIAUCSL.csv"), "observation_date", "CPIAUCSL", "MS")

adjusted_df = apply_calendar_and_price_adjustments(raw_sales, cpi_series)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(adjusted_df.index, adjusted_df["raw_sales"] / 1e3, label="Nominal Sales ($B)", color="#1F77B4", lw=1.5)
axes[0].plot(adjusted_df.index, adjusted_df["real_sales"] / 1e3, label="Real Sales (2026 Dollar Base, $B)", color="#2CA02C", lw=1.5)
axes[0].set_ylabel("Sales ($ Billions)")
axes[0].set_title("US Retail Sales: Nominal vs Inflation-Deflated (Real) Trajectory", fontsize=13, fontweight='bold')
axes[0].legend(loc="upper left")

axes[1].plot(adjusted_df.index, adjusted_df["real_sales_per_day"], label="Real Sales per Day ($M/day)", color="#D62728", lw=1.2)
axes[1].set_ylabel("Real Sales / Day ($M)")
axes[1].set_title("Calendar-Normalized Consumption Rate (Leap Year & Month-Length Corrected)", fontsize=13, fontweight='bold')
axes[1].legend(loc="upper left")

plt.tight_layout()
plt.show()

print(f"Sample adjustment table (head):\n{adjusted_df.head(3)}")


---
## Part 2: Variance Stabilization & The Guerrero Method (20 Minutes)

### Theoretical Foundations
Many time series exhibit **heteroskedasticity**, where the seasonal or random variation increases proportionally with the overall level of the series. Standard linear models assume homoskedastic errors $\epsilon_t \sim \mathcal{N}(0, \sigma^2)$.

The **Box-Cox transformation family** stabilizes variance:
$$w_t = \begin{cases} \frac{y_t^\lambda - 1}{\lambda} & \text{if } \lambda \neq 0 \\ \ln(y_t) & \text{if } \lambda = 0 \end{cases}$$

#### The Guerrero (1993) Method
Rather than assuming Gaussian residuals via profile likelihood, Guerrero (1993) proposed dividing the series into non-overlapping seasonal blocks of length $m$. For each block $i$, let $m_i$ be the mean and $s_i$ be the standard deviation. Under a Box-Cox transform, the local standard deviation is approximated by:
$$w_i(\lambda) = \frac{s_i}{m_i^{1 - \lambda}}$$
Guerrero selects $\lambda$ to minimize the coefficient of variation (CV) of $w_i(\lambda)$ across all blocks:
$$\text{CV}(\lambda) = \frac{\text{std}(w_i)}{\text{mean}(w_i)}$$

#### Inversion & Bias Correction (Jensen's Inequality)
Because the Box-Cox inverse is non-linear, naive inversion of a mean forecast on the transformed scale underestimates the true expected value on the original scale:
$$\mathbb{E}[y] > g^{-1}(\mathbb{E}[w])$$
Using a second-order Taylor series expansion, the **bias-corrected mean forecast** is:
$$\hat{y} \approx (w\lambda + 1)^{1/\lambda} \left[ 1 + \frac{\sigma^2 (1 - \lambda)}{2 (w\lambda + 1)^2} \right] \quad (\lambda \neq 0)$$
$$\hat{y} \approx \exp(w) \left[ 1 + \frac{\sigma^2}{2} \right] \quad (\lambda = 0)$$

### Implementation Task 2
Implement `estimate_guerrero_lambda`, `apply_boxcox`, and `invert_boxcox_mean`.


In [ ]:
def estimate_guerrero_lambda(
    series: pd.Series,
    season_length: int = 12,
    lower: float = -1.0,
    upper: float = 2.0
) -> float:
    """
    Estimates optimal Box-Cox transformation parameter lambda using Guerrero's (1993) method.
    Minimizes the coefficient of variation of standard deviations divided by the mean^(1 - lambda)
    across non-overlapping seasonal blocks.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


def apply_boxcox(series: pd.Series, lmbda: float) -> pd.Series:
    """
    Applies the mathematical Box-Cox transformation to a positive series.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


def invert_boxcox_mean(
    transformed: pd.Series,
    lmbda: float,
    sigma2: Optional[float] = None
) -> pd.Series:
    """
    Inverts Box-Cox transformation with optional second-order Taylor series bias correction.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 2: Guerrero Lambda Optimization & Variance Stabilization
opt_lambda = estimate_guerrero_lambda(raw_sales, season_length=12)
print(f"Optimal Guerrero lambda for US Retail Sales: {opt_lambda:.4f}")

# Evaluate Guerrero CV profile over grid
grid = np.linspace(-0.5, 1.5, 100)
cv_values = []
k = len(raw_sales) // 12
blocks = raw_sales.values[:k*12].reshape(k, 12)
means = np.mean(blocks, axis=1)
stds = np.std(blocks, axis=1, ddof=1)
for l in grid:
    w = stds / (means ** (1.0 - l))
    cv_values.append(np.std(w, ddof=1) / np.mean(w))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(grid, cv_values, color="#1F77B4", lw=2, label="Guerrero CV Profile")
ax.axvline(opt_lambda, color="#D62728", ls="--", lw=1.5, label=f"Optimal $\lambda^* = {opt_lambda:.4f}$")
ax.set_xlabel("Transformation Parameter $\lambda$")
ax.set_ylabel("Coefficient of Variation (CV)")
ax.set_title("Guerrero (1993) Objective Function Minimization", fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

# Verification of inversion roundtrip
bc_series = apply_boxcox(raw_sales, opt_lambda)
rec_series = invert_boxcox_mean(bc_series, opt_lambda)
max_inversion_err = np.max(np.abs(raw_sales - rec_series))
print(f"Max numerical inversion discrepancy: {max_inversion_err:.2e} (Machine Precision)")


---
## Part 3: Classical Decomposition & Centered Moving Averages (20 Minutes)

### Theoretical Foundations
A moving average filter of order $m$ smooths high-frequency seasonal oscillations:
$$\hat{T}_t = \frac{1}{m} \sum_{j=-k}^k y_{t+j}$$
When $m$ is even (such as $m=12$ for monthly data), a simple $m$-MA is asymmetric relative to the integer time index, falling at half-integer intervals ($t + 0.5$).
To center the moving average on the integer observation points, a **$2 \times m$-MA** is required:
$$\hat{T}_t = \frac{1}{2m} y_{t - m/2} + \frac{1}{m} y_{t - m/2 + 1} + \dots + \frac{1}{m} y_{t + m/2 - 1} + \frac{1}{2m} y_{t + m/2}$$
For $m=12$, this produces convolution weights:
$$\left[ \frac{1}{24}, \frac{1}{12}, \frac{1}{12}, \dots, \frac{1}{12}, \frac{1}{24} \right]$$

#### The Endpoint Void Problem
Because the centered moving average requires $m/2$ observations before and $m/2$ observations after time $t$:
- Exactly $m/2$ observations at the start ($t=1..m/2$) are missing (`NaN`).
- Exactly $m/2$ observations at the end ($t=T - m/2 + 1..T$) are missing (`NaN`).
This structural void at the forecast origin ($t=T$) makes classical moving averages unusable for real-time forecasting.

### Implementation Task 3
Implement `compute_centered_ma`.


In [ ]:
def compute_centered_ma(series: pd.Series, period: int = 12) -> pd.DataFrame:
    """
    Computes classical centered moving average (2 x m MA for even m, m-MA for odd m).
    Leaves exactly m endpoint voids as NaN to prevent edge distortion.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 3: Centered Moving Averages & Endpoint Voids
ma_df = compute_centered_ma(adjusted_df["real_sales"], period=12)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(ma_df.index, ma_df["raw"] / 1e3, label="Real Sales ($B)", color="#7F7F7F", alpha=0.6, lw=1)
axes[0].plot(ma_df.index, ma_df["centered_trend"] / 1e3, label="2x12 Centered MA Trend", color="#1F77B4", lw=2)
# Highlight the endpoint void at the end of the series
axes[0].axvspan(ma_df.index[-6], ma_df.index[-1], color="#FF7F0E", alpha=0.25, label="Right Endpoint Void (6 Months NaN)")
axes[0].set_ylabel("Sales ($ Billions)")
axes[0].set_title("Classical 2x12 Centered Moving Average Trend Filter", fontsize=12, fontweight='bold')
axes[0].legend(loc="upper left")

axes[1].plot(ma_df.index, ma_df["detrended_ratio"], color="#2CA02C", lw=1.2, label="Multiplicative Detrended Ratio ($Y_t / \hat{T}_t$)")
axes[1].axhline(1.0, color="black", ls="--", lw=0.8)
axes[1].set_ylabel("Seasonal Multiplier")
axes[1].set_title("Detrended Multiplicative Seasonal Ratio", fontsize=12, fontweight='bold')
axes[1].legend(loc="upper left")

plt.tight_layout()
plt.show()

n_start_nans = ma_df["centered_trend"].iloc[:6].isna().sum()
n_end_nans = ma_df["centered_trend"].iloc[-6:].isna().sum()
print(f"Verified Endpoint Voids: {n_start_nans} leading NaNs, {n_end_nans} trailing NaNs.")


---
## Part 4: Modern Nonparametric Decomposition: Multiplicative Log-STL (25 Minutes)

### Theoretical Foundations
**STL** (*Seasonal-Trend Decomposition using LOESS*) addresses the severe limitations of classical moving averages:
- It handles complex non-linear trends via **LOESS** (locally estimated scatterplot smoothing).
- Seasonality can evolve flexibly over time rather than remaining strictly fixed.
- Outlier robustness is achieved via iterative bi-square reweighting in the outer loop.
- **Zero endpoint voids:** Because LOESS fits local polynomials, it extrapolates smoothly to the boundary observations $t=1$ and $t=T$.

### Direction 1: Multiplicative Log-STL Formulation
Standard STL is inherently **additive**:
$$y_t = T_t + S_t + R_t$$
For economic and retail series where seasonal amplitude scales proportionally with the trend, a **multiplicative decomposition** is required:
$$Y_t = T_t \times S_t \times R_t$$
We formulate **Multiplicative Log-STL**:
1. Take the natural logarithm: $y_t^* = \ln(Y_t)$
2. Fit STL on $y_t^*$:
   $$y_t^* = T_t^* + S_t^* + R_t^*$$
3. Exponentiate each component back to the original scale:
   $$T_t = \exp(T_t^*), \quad S_t = \exp(S_t^*), \quad R_t = \exp(R_t^*)$$
4. Multiplicative reconstruction holds identically:
   $$T_t \times S_t \times R_t = \exp(T_t^* + S_t^* + R_t^*) = \exp(y_t^*) = Y_t$$
5. Seasonally adjusted series:
   $$A_t = \frac{Y_t}{S_t} = T_t \times R_t = \exp(T_t^* + R_t^*)$$

### Implementation Task 4
Implement `decompose_log_stl`.


In [ ]:
def decompose_log_stl(
    series: pd.Series,
    period: int = 12,
    seasonal: int = 13,
    robust: bool = True,
    seasonal_deg: int = 0,
    trend_deg: int = 1
) -> pd.DataFrame:
    """
    Direction 1: Multiplicative Log-STL Decomposition.
    Applies STL to log-transformed series and exponentiates back to guarantee
    exact multiplicative reconstruction: Y_t = T_t * S_t * R_t.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 4: Multiplicative Log-STL Four-Panel Dashboard
log_stl_df = decompose_log_stl(raw_sales, period=12, seasonal=13, robust=True)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

axes[0].plot(log_stl_df.index, log_stl_df["observed"] / 1e3, color="#1F77B4", lw=1.5)
axes[0].set_ylabel("Observed ($B)")
axes[0].set_title("Multiplicative Log-STL Decomposition: US Retail Sales ($Y_t = T_t \times S_t \times R_t$)", fontsize=13, fontweight='bold')

axes[1].plot(log_stl_df.index, log_stl_df["trend"] / 1e3, color="#2CA02C", lw=2)
axes[1].set_ylabel("Trend ($B)")

axes[2].plot(log_stl_df.index, log_stl_df["seasonal"], color="#D62728", lw=1.2)
axes[2].axhline(1.0, color="black", ls="--", lw=0.8)
axes[2].set_ylabel("Seasonal Factor ($S_t$)")

axes[3].plot(log_stl_df.index, log_stl_df["remainder"], color="#9467BD", lw=1)
axes[3].axhline(1.0, color="black", ls="--", lw=0.8)
axes[3].set_ylabel("Remainder ($R_t$)")

plt.tight_layout()
plt.show()

# Verification of Multiplicative Identity
recon = log_stl_df["trend"] * log_stl_df["seasonal"] * log_stl_df["remainder"]
max_rel_err = np.max(np.abs((raw_sales - recon) / raw_sales))
print(f"Maximum Relative Multiplicative Discrepancy: {max_rel_err:.2e} (Exact Identity)")


---
## Part 5: Time Series Feature Engineering (FEASTS Architecture) (15 Minutes)

### Theoretical Foundations
In large-scale forecasting systems with thousands of series, inspecting visual plots manually is impossible. The **FEASTS** framework (Wang, Smith & Hyndman, 2006) extracts scalar statistical features from decompositions:

#### 1. Trend Strength ($F_T$)
$$F_T = \max\left(0, 1 - \frac{\text{Var}(R_t)}{\text{Var}(T_t + R_t)}\right)$$
If a series has little to no trend, $\text{Var}(T + R) \approx \text{Var}(R)$, yielding $F_T \approx 0$. If trend dominates remainder noise, $F_T \to 1$.

#### 2. Seasonal Strength ($F_S$)
$$F_S = \max\left(0, 1 - \frac{\text{Var}(R_t)}{\text{Var}(S_t + R_t)}\right)$$
Measures the proportion of detrended variance explained by cyclical seasonal factors.

#### 3. Peak & Trough Timing
Calendar month indices (1..12) where seasonal multiplier achieves maximum and minimum.

#### 4. Remainder Diagnostics
Lag-1 autocorrelation $r_1(R_t)$, skewness, and kurtosis to identify white noise departure and extreme structural shocks (e.g. COVID-19).

### Implementation Task 5
Implement `extract_feasts_features`.


In [ ]:
def extract_feasts_features(
    decomposition_df: pd.DataFrame,
    is_multiplicative: bool = False
) -> Dict[str, float]:
    """
    Extracts FEASTS-style summary metrics from a time series decomposition.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 5: FEASTS Feature Comparison across Macro Series
indpro_series = load_and_index_series(str(DATA_DIR / "INDPRO.csv"), "observation_date", "INDPRO", "MS")
indpro_decomp = decompose_log_stl(indpro_series, period=12)

features_retail = extract_feasts_features(log_stl_df, is_multiplicative=True)
features_indpro = extract_feasts_features(indpro_decomp, is_multiplicative=True)

feats_comparison = pd.DataFrame([features_retail, features_indpro], index=["Retail Sales (RSAFS)", "Industrial Production (INDPRO)"])
print("Comparative FEASTS Feature Profiles:")
print(feats_comparison.T)


---
## Part 6: Production STL Forecaster (STLF) & Temporal Leakage Auditing (20 Minutes)

### Theoretical Foundations
Decomposition provides one of the most reliable forecasting architectures in industrial production (**STLF**):
1. **Decompose:** $Y_{1:T} = T_t \times S_t \times R_t$.
2. **Seasonally Adjusted Series:** $A_t = Y_t / S_t$. This series is free of seasonal oscillations and can be reliably projected into the future using non-seasonal methods (such as Random Walk with Drift):
   $$\hat{A}_{T+h} = A_T + h \left( \frac{A_T - A_1}{T - 1} \right)$$
3. **Seasonal Projection:** Project seasonal factors using Seasonal Naïve (repeating the most recent observed cycle):
   $$\hat{S}_{T+h} = S_{T + h - m}$$
4. **Recombine:**
   $$\hat{Y}_{T+h} = \hat{A}_{T+h} \times \hat{S}_{T+h}$$

### Temporal Hygiene & Leakage Auditing
Time series models must never leak future information backwards:
- **Train/Test Splitting:** Must be strictly causal ($t_{\text{train, max}} < t_{\text{test, min}}$).
- **Imputation:** Bi-directional methods (`interpolate`, `bfill`) leak future target levels into past missing values, generating artificially optimistic backtest performance. Only causal forward-filling (`ffill`) is causally valid.

### Implementation Task 6
Implement `audit_table_shift_leakage`, `evaluate_imputation_leakage`, and `forecast_stl_decomposition`.


In [ ]:
def audit_table_shift_leakage(
    train_series: pd.Series,
    test_series: pd.Series
) -> Dict[str, Any]:
    """
    Audits temporal hygiene across train and test splits to guarantee no lookahead data leakage.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


def evaluate_imputation_leakage(series_with_nans: pd.Series) -> Dict[str, pd.Series]:
    """
    Demonstrates data leakage caused by bi-directional imputation versus causal forward-filling.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


def forecast_stl_decomposition(
    series: pd.Series,
    horizon: int = 12,
    period: int = 12,
    robust: bool = True
) -> Dict[str, pd.Series]:
    """
    Direction 4: Production STLF Decomposition Forecaster.
    Combines Log-STL, seasonally adjusted drift, and seasonal naive projection.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 6: STLF 12-Month Out-of-Sample Forecast
fc_res = forecast_stl_decomposition(raw_sales, horizon=12, period=12)

fig, ax = plt.subplots(figsize=(12, 5))
recent_history = raw_sales.iloc[-48:]
ax.plot(recent_history.index, recent_history / 1e3, label="Historical Actuals", color="#1F77B4", lw=1.8)
ax.plot(fc_res["forecast"].index, fc_res["forecast"] / 1e3, label="STLF 12-Month Forecast", color="#D62728", lw=2, marker='o', markersize=4)
ax.plot(fc_res["trend_forecast"].index, fc_res["trend_forecast"] / 1e3, label="Seasonally Adjusted Drift ($A_{T+h}$)", color="#2CA02C", ls="--", lw=1.5)

ax.set_ylabel("Sales ($ Billions)")
ax.set_title("Production STLF Forecast: Recombined Seasonally Adjusted Drift + Seasonal Naive", fontsize=13, fontweight='bold')
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

print(f"Generated {len(fc_res['forecast'])} out-of-sample forecasts from {fc_res['forecast'].index[0].date()} to {fc_res['forecast'].index[-1].date()}.")


---
## Part 7: Conceptual & Quantitative Interview Questions (10 Points)

### Question 1: Multiplicative Dynamics & Jensen's Inequality
When is a multiplicative decomposition model mathematically necessary compared to an additive model? Furthermore, when forecasting on the logarithmic scale ($\hat{w}_{T+h}$), why does naive exponentiation $\hat{y}_{T+h} = \exp(\hat{w}_{T+h})$ produce an underestimate of the expected mean $\mathbb{E}[y_{T+h}]$, and how does the Taylor series correction resolve this?


*Write your answer here in markdown...*


### Question 2: Moving Average Endpoint Voids & The STL Advantage
Explain why the classical $2 \times 12$ centered moving average filter cannot produce an estimate of the trend at the most recent observation $T$, creating an "endpoint void". Why is this fatal for real-time operational forecasting, and how does the LOESS polynomial regression in STL overcome this restriction?


*Write your answer here in markdown...*


### Question 3: Data Leakage in Time Series Feature Engineering & Preprocessing
Describe two distinct ways a practitioner might inadvertently introduce lookahead data leakage when engineering features or imputing missing values for a time series model. What strict temporal hygiene rules must be enforced during cross-validation?


*Write your answer here in markdown...*
